# Módulo 1 · Etapa 3 — Inventario de artefactos
**TFM · HAM10000** · Responsable: **Adrián Arias** · MLflow: experimento `m1-datos`, run `m1-artefactos`, tag `fase=artefactos`

**Objetivo:** medir qué elementos ajenos a la lesión aparecen en cada clase. Son los candidatos a **atajos espurios** que buscará el Módulo 3 con Grad-CAM (que el modelo acierte mirando la regla o el borde negro en vez de la lesión).

**Cómo está organizado (se puede hacer en varias sesiones):**

| Parte | Secciones | Duración | Cuándo |
|---|---|---|---|
| A · Preparación | 1–4 | ~5 min | Una vez |
| B · Etiquetado manual | 5 | 2–3 h (350 imágenes) | En las sesiones que necesites: guarda a cada clic |
| C · Resultados | 6–9 | ~5 min | Cuando termines de etiquetar |

**Decisiones de diseño**
- **Muestra solo de train** (nunca de validación ni de prueba): 50 lesiones distintas por clase, una foto por lesión, semilla 42. En `df` train tiene justo 50 lesiones, así que entran todas.
- **Etiquetado a ciegas:** la herramienta **no muestra la clase**, y el orden está mezclado. Así la etiqueta no se contagia de lo que esperas ver en cada diagnóstico.
- **Definiciones operativas fijas** (se ven en la herramienta): si dudas, aplica la definición literal y anota la duda en el comentario.
- **Acuerdo entre anotadores (opcional):** otro integrante etiqueta 42 de las mismas imágenes (6 por clase) y se calcula el kappa de Cohen.

Entorno: **CPU**. Secretos: `DAGSHUB_USER`, `DAGSHUB_TOKEN`.

# Parte A · Preparación
## 1. Sesión

In [ ]:
ANOTADOR = "adrian"          # quien etiqueta en esta sesión: adrian | christian | alex
SOLO_ACUERDO = False         # True solo para el segundo anotador (etiqueta 42 imágenes, no las 350)
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
DATASET_SHA256_ESPERADO = "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5"
SPLIT_SHA256_ESPERADO = "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38"   # split v1 (scikit-learn 1.9.1)

In [ ]:
import os, json, time, shutil, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/TFM")

REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import pandas as pd, numpy as np
from src.tracking import load_config, file_sha256
cfg = load_config("configs/base.yaml")

# Imágenes al disco local (rápido)
LOCAL = Path(cfg["paths"]["local_data_dir"])
if not LOCAL.exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time()-t0:.0f} s)")
IMAGES, META = Path(cfg["paths"]["images_dir"]), Path(cfg["paths"]["metadata_csv"])
assert file_sha256(META) == DATASET_SHA256_ESPERADO, "[!] Metadatos distintos a los de la Etapa 0"

# Split oficial desde Drive, verificado por su huella
SPLIT_DRIVE = DRIVE_ROOT / "data" / "splits" / "split_v1.csv"
assert file_sha256(SPLIT_DRIVE) == SPLIT_SHA256_ESPERADO, "[!] El split de Drive no es el split v1 publicado."
split, meta = pd.read_csv(SPLIT_DRIVE), pd.read_csv(META)
print("[+] Dataset y split v1 verificados ·", split.split.value_counts().to_dict())

OUT = DRIVE_ROOT / "results" / "m1" / "etapa3"; OUT.mkdir(parents=True, exist_ok=True)
FIG = DRIVE_ROOT / "figures" / "m1" / "etapa3"; FIG.mkdir(parents=True, exist_ok=True)

## 2. Funciones

In [ ]:
# --- Lógica de la Etapa 3 ---
import io
import numpy as np
import pandas as pd
from pathlib import Path

CLASES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
ARTEFACTOS = {   # clave: (nombre corto, definición operativa usada al etiquetar)
    "pelo":       ("Pelo", "Al menos un pelo cruza o toca la lesión."),
    "regla":      ("Regla / escala", "Regla, marcas de graduación o escala visibles en cualquier parte."),
    "tinta":      ("Marca de tinta", "Trazo de rotulador o tinta sobre la piel (círculo, punto, flecha)."),
    "borde":      ("Borde negro", "Zona negra curva en una o más esquinas por el campo circular del dermatoscopio."),
    "burbujas":   ("Burbujas", "Burbujas de gel o de aire visibles."),
    "reflejos":   ("Reflejos", "Brillos blancos especulares (zonas quemadas por la luz)."),
}

def muestra_estratificada(split, meta, por_clase=50, semilla=42):
    """50 lesiones distintas por clase, SOLO de train, una imagen al azar por lesión.
    Se baraja el orden final para que las clases se mezclen al etiquetar."""
    tr = split[split.split == "train"].merge(meta[["image_id", "dx_type"]], on="image_id")
    una = tr.sample(frac=1, random_state=semilla).drop_duplicates("lesion_id")   # una foto por lesión
    partes = []
    for c in CLASES:
        sub = una[una.dx == c]
        partes.append(sub.sample(min(por_clase, len(sub)), random_state=semilla))
    m = pd.concat(partes).sample(frac=1, random_state=semilla).reset_index(drop=True)
    m.insert(0, "orden", range(1, len(m) + 1))
    return m[["orden", "image_id", "lesion_id", "dx", "dx_type"]]

def cargar_etiquetas(ruta, muestra):
    """Lee el CSV de etiquetas (si existe) y lo alinea con la muestra."""
    cols = ["image_id"] + list(ARTEFACTOS) + ["comentario", "anotador", "fecha"]
    if Path(ruta).exists():
        et = pd.read_csv(ruta, dtype={"comentario": str}).fillna({"comentario": ""})
    else:
        et = pd.DataFrame(columns=cols)
    return et[et.image_id.isin(muestra.image_id)]

def guardar_etiqueta(ruta, fila):
    """Añade o reemplaza la etiqueta de una imagen y guarda el CSV completo (seguro ante cortes)."""
    ruta = Path(ruta)
    et = pd.read_csv(ruta, dtype={"comentario": str}).fillna({"comentario": ""}) if ruta.exists() else pd.DataFrame()
    if len(et):
        et = et[et.image_id != fila["image_id"]]
    et = pd.concat([et, pd.DataFrame([fila])], ignore_index=True)
    tmp = ruta.with_suffix(".tmp")
    et.to_csv(tmp, index=False)
    tmp.replace(ruta)                      # escritura atómica: nunca queda un CSV a medias
    return len(et)

def wilson(k, n, z=1.96):
    """Intervalo de confianza del 95 % de una proporción (Wilson), en %."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (100 * max(0, c - h), 100 * min(1, c + h))

def tabla_por_clase(etiq):
    """% de imágenes con cada artefacto, por clase, con IC 95 % y comparación frente a nv (Fisher)."""
    from scipy.stats import fisher_exact
    filas = []
    for a in ARTEFACTOS:
        nv = etiq[etiq.dx == "nv"][a]
        for c in CLASES:
            s = etiq[etiq.dx == c][a].astype(int)
            k, n = int(s.sum()), len(s)
            lo, hi = wilson(k, n)
            p = np.nan
            if c != "nv" and n and len(nv):
                p = fisher_exact([[k, n - k], [int(nv.sum()), len(nv) - int(nv.sum())]])[1]
            filas.append(dict(artefacto=a, clase=c, n=n, con_artefacto=k,
                              pct=round(100 * k / n, 1) if n else np.nan,
                              ic95_inf=round(lo, 1), ic95_sup=round(hi, 1), p_vs_nv=p))
    t = pd.DataFrame(filas)
    nv_pct = t[t.clase == "nv"].set_index("artefacto").pct
    t["dif_vs_nv_pp"] = (t.pct - t.artefacto.map(nv_pct)).round(1)
    t["senal"] = (t.clase != "nv") & (t.p_vs_nv < 0.05) & (t.dif_vs_nv_pp.abs() >= 15)
    return t

def acuerdo(etq_a, etq_b):
    """Kappa de Cohen por artefacto entre dos anotadores, sobre las imágenes que ambos etiquetaron."""
    from sklearn.metrics import cohen_kappa_score
    comun = etq_a.merge(etq_b, on="image_id", suffixes=("_a", "_b"))
    res = {}
    for a in ARTEFACTOS:
        x, y = comun[f"{a}_a"].astype(int), comun[f"{a}_b"].astype(int)
        res[a] = dict(n=len(comun), acuerdo_pct=round(100 * (x == y).mean(), 1) if len(comun) else np.nan,
                      kappa=round(cohen_kappa_score(x, y), 2) if len(comun) and (x.nunique() > 1 or y.nunique() > 1) else np.nan)
    return pd.DataFrame(res).T

def validar_regla_borde(etiq, color, umbral=40):
    """Compara la regla automática 'brillo de esquinas < umbral' con la etiqueta manual de borde negro."""
    m = etiq[["image_id", "borde"]].merge(color[["image_id", "esquinas"]], on="image_id")
    pred, real = (m.esquinas < umbral).astype(int), m.borde.astype(int)
    tp, fp = int(((pred == 1) & (real == 1)).sum()), int(((pred == 1) & (real == 0)).sum())
    fn, tn = int(((pred == 0) & (real == 1)).sum()), int(((pred == 0) & (real == 0)).sum())
    return dict(umbral=umbral, n=len(m), vp=tp, fp=fp, fn=fn, vn=tn,
                precision=round(tp / (tp + fp), 2) if tp + fp else np.nan,
                sensibilidad=round(tp / (tp + fn), 2) if tp + fn else np.nan)

In [ ]:
# --- Herramienta de etiquetado (ipywidgets, funciona en Colab) ---
import io, datetime
import ipywidgets as W
from PIL import Image

def crear_etiquetador(muestra, ruta_csv, images_dir, anotador, ids=None):
    """Muestra las imágenes UNA a UNA, SIN la clase (etiquetado a ciegas), con una casilla por artefacto.
    Cada 'Guardar y siguiente' escribe en Drive: si Colab se corta, se retoma en la primera sin etiquetar."""
    lista = muestra if ids is None else muestra[muestra.image_id.isin(ids)]
    lista = lista.reset_index(drop=True)
    hechas = set(cargar_etiquetas(ruta_csv, lista).image_id)
    pendientes = [i for i, iid in enumerate(lista.image_id) if iid not in hechas]
    estado = {"i": pendientes[0] if pendientes else len(lista) - 1}

    img_w = W.Image(format="jpeg", layout=W.Layout(width="600px", height="450px"))
    casillas = {a: W.Checkbox(description=nom, value=False, indent=False,
                              layout=W.Layout(width="180px"))
                for a, (nom, defi) in ARTEFACTOS.items()}
    comentario = W.Text(placeholder="Comentario (opcional)", layout=W.Layout(width="560px"))
    progreso, info = W.HTML(), W.HTML()
    b_ant, b_sig, b_salt = W.Button(description="◀ Anterior"), W.Button(description="Guardar y siguiente ▶", button_style="success"), W.Button(description="Ir a la siguiente pendiente")

    def mostrar():
        i = estado["i"]; iid = lista.image_id[i]
        with Image.open(Path(images_dir) / f"{iid}.jpg") as im:
            buf = io.BytesIO(); im.convert("RGB").save(buf, "JPEG", quality=90); img_w.value = buf.getvalue()
        previas = cargar_etiquetas(ruta_csv, lista).set_index("image_id")
        for a, cb in casillas.items():
            cb.value = bool(previas.loc[iid, a]) if iid in previas.index else False
        comentario.value = str(previas.loc[iid, "comentario"]) if iid in previas.index else ""
        n_hechas = len(previas)
        progreso.value = f"<b>Imagen {i + 1} de {len(lista)}</b> · {iid} · etiquetadas: {n_hechas}/{len(lista)}" + (
            " · <span style='color:green'>ya etiquetada (puedes corregirla)</span>" if iid in previas.index else "")

    def guardar_y_seguir(_):
        iid = lista.image_id[estado["i"]]
        fila = {"image_id": iid, **{a: int(cb.value) for a, cb in casillas.items()},
                "comentario": comentario.value.strip(), "anotador": anotador,
                "fecha": datetime.datetime.now().isoformat(timespec="seconds")}
        n = guardar_etiqueta(ruta_csv, fila)
        if estado["i"] < len(lista) - 1:
            estado["i"] += 1; mostrar(); info.value = ""
        else:
            mostrar(); info.value = f"<b style='color:green'>✅ Última imagen guardada. Total: {n}.</b>"

    def anterior(_):
        if estado["i"] > 0:
            estado["i"] -= 1; mostrar()

    def saltar(_):
        hechas = set(cargar_etiquetas(ruta_csv, lista).image_id)
        pend = [i for i, iid in enumerate(lista.image_id) if iid not in hechas]
        if pend:
            estado["i"] = pend[0]; mostrar()
        else:
            info.value = "<b style='color:green'>✅ No queda ninguna pendiente.</b>"

    b_sig.on_click(guardar_y_seguir); b_ant.on_click(anterior); b_salt.on_click(saltar)
    leyenda = W.HTML("<br>".join(f"<b>{nom}:</b> {defi}" for nom, defi in ARTEFACTOS.values()))
    definiciones = W.Accordion(children=[leyenda]); definiciones.set_title(0, "Definiciones")
    mostrar()
    return W.VBox([progreso, img_w,
                   W.HBox(list(casillas.values())[:3]), W.HBox(list(casillas.values())[3:]),
                   comentario, W.HBox([b_ant, b_sig, b_salt]), info, definiciones])

In [ ]:
import matplotlib.pyplot as plt
AZUL, TINTA, TINTA2 = "#2a78d6", "#1f1f1d", "#6b6a64"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.edgecolor": "#c9c8c1",
                     "axes.labelcolor": TINTA2, "xtick.color": TINTA2, "ytick.color": TINTA2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e8e7e2", "axes.axisbelow": True, "legend.frameon": False})
def es(x, dec=1):
    return f"{x:,.{dec}f}".replace(",", "X").replace(".", ",").replace("X", ".")
def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", bbox_inches="tight", facecolor="white"); plt.show()

## 3. Muestra estratificada de train
Se genera **una sola vez** y se guarda en Drive. Si ya existe, se reutiliza (así todos los anotadores ven las mismas imágenes).

In [ ]:
MUESTRA_CSV = OUT / "muestra_artefactos.csv"
if MUESTRA_CSV.exists():
    muestra = pd.read_csv(MUESTRA_CSV); print("Muestra existente reutilizada")
else:
    muestra = muestra_estratificada(split, meta, por_clase=50, semilla=42)
    muestra.to_csv(MUESTRA_CSV, index=False); print("[+] Muestra creada y guardada en", MUESTRA_CSV)

assert set(muestra.image_id) <= set(split[split.split == "train"].image_id), "[!] La muestra tiene imágenes fuera de train"
assert muestra.lesion_id.is_unique
print(len(muestra), "imágenes ·", muestra.dx.value_counts().reindex(CLASES).to_dict())

# Subconjunto para el segundo anotador: las 6 primeras de cada clase según el orden mezclado
ACUERDO_IDS = muestra.groupby("dx").head(6).image_id.tolist()
print("Subconjunto de acuerdo:", len(ACUERDO_IDS), "imágenes")

## 4. Color de `nv` según el método de confirmación
En la Etapa 2 se observó que `follow_up` **solo aparece en `nv`** (55 % de la clase). Si esas imágenes se tomaron con otro equipo o protocolo, se verían distintas, y el modelo podría aprender "cómo se capturó la foto" en lugar de la lesión.

Se compara, **solo en train**, el color de las imágenes de `nv` confirmadas por seguimiento frente a las confirmadas por histopatología. Se reutilizan las medidas de color de la Etapa 2.

In [ ]:
color = pd.read_csv(DRIVE_ROOT / "results" / "m1" / "etapa2" / "color_por_imagen.csv")
nv_tr = (split[(split.split == "train") & (split.dx == "nv")]
         .merge(meta[["image_id", "dx_type"]], on="image_id").merge(color.drop(columns="dx"), on="image_id"))
VARS = ["brillo", "R", "G", "B", "esquinas"]
res = nv_tr.groupby("dx_type")[VARS].median().round(1)
res.insert(0, "n", nv_tr.dx_type.value_counts())
ref = nv_tr[nv_tr.dx_type == "histo"]
efecto = {t: {v: round(float((g[v].median() - ref[v].median()) / ref[v].std()), 2) for v in VARS}
          for t, g in nv_tr.groupby("dx_type") if t != "histo"}
res.to_csv(OUT / "color_nv_por_dx_type.csv"); display(res)
print("Diferencia de mediana frente a nv-histo, en desviaciones típicas:")
for t, d in efecto.items(): print(f"  {t}: {d}")
print("Referencia: |d| < 0,2 despreciable · ~0,5 moderada · > 0,8 grande")

orden = [t for t in ["histo", "consensus", "follow_up"] if t in nv_tr.dx_type.unique()]
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), sharey=True)
for ax, v, xl in zip(axes, ["brillo", "R", "esquinas"], ["Brillo medio", "Canal rojo", "Brillo de esquinas"]):
    ax.boxplot([nv_tr.loc[nv_tr.dx_type == t, v] for t in orden], vert=False, tick_labels=orden, widths=0.55,
               patch_artist=True, showfliers=False, boxprops=dict(facecolor="#cde2fb", edgecolor=AZUL),
               medianprops=dict(color=AZUL, linewidth=2), whiskerprops=dict(color=AZUL), capprops=dict(color=AZUL))
    ax.set_xlabel(f"{xl} (0–255)"); ax.set_xlim(0, 255); ax.grid(axis="y", visible=False)
fig.tight_layout(); guardar(fig, "fig_color_nv_por_dx_type")

# Parte B · Etiquetado manual
## 5. Herramienta de etiquetado
**Cómo usarla**
1. Ejecuta la celda. Aparece una imagen y seis casillas.
2. Marca **solo** los artefactos que veas, según su definición (despliega *Definiciones* abajo). Si no hay ninguno, no marques nada.
3. Pulsa **Guardar y siguiente**. Cada clic se guarda en Drive (`etapa3/etiquetas_<anotador>.csv`).
4. Puedes parar cuando quieras. Al volver, ejecuta las secciones 1–3 y esta celda: continúa en la primera pendiente.
5. **◀ Anterior** permite corregir una ya guardada (se reemplaza, no se duplica).

Consejo: haz bloques de 50–70 imágenes con descansos; el cansancio baja la calidad del etiquetado.

**Segundo anotador** (opcional, recomendado): otro integrante abre este mismo notebook desde su cuenta, con acceso a la carpeta `TFM/` compartida, pone `ANOTADOR = "christian"` (o `"alex"`) y `SOLO_ACUERDO = True` en la sección 1. Etiqueta 42 imágenes sin ver las tuyas. Ejecuta **solo las secciones 1, 2, 3 y 5** (no la 4 ni la Parte C).

In [ ]:
ETIQ_CSV = OUT / f"etiquetas_{ANOTADOR}.csv"
crear_etiquetador(muestra, ETIQ_CSV, IMAGES, ANOTADOR, ids=ACUERDO_IDS if SOLO_ACUERDO else None)

# Parte C · Resultados
Ejecutar al terminar las 350 imágenes. Si la sesión de Colab es nueva (se desconectó o es otro día), ejecuta antes las secciones 1–3; si es la misma sesión, no hace falta.

Las secciones 6 y 8 se pueden ejecutar ya. La **7 y la 9, solo cuando el segundo anotador haya terminado** (o cuando se decida que no habrá segundo anotador): la 9 registra el run en MLflow y conviene hacerlo una sola vez.

## 6. Porcentaje de cada artefacto por clase

In [ ]:
et = cargar_etiquetas(OUT / "etiquetas_adrian.csv", muestra)
faltan = len(muestra) - len(et)
assert faltan == 0, f"[!] Faltan {faltan} imágenes por etiquetar (sección 5)"
etiq = muestra.merge(et, on="image_id")
for a in ARTEFACTOS: etiq[a] = etiq[a].astype(int)
etiq.to_csv(OUT / "artefactos_muestra.csv", index=False)

tab = tabla_por_clase(etiq)
tab.to_csv(OUT / "artefactos_por_clase.csv", index=False)
pct = tab.pivot(index="artefacto", columns="clase", values="pct").reindex(index=list(ARTEFACTOS), columns=CLASES)
display(pct)

print("Global en la muestra (%):", {a: es(100 * etiq[a].mean()) for a in ARTEFACTOS})
s = tab[tab.senal].sort_values("dif_vs_nv_pp", key=abs, ascending=False)
print("\nDiferencias relevantes frente a nv (p < 0,05 y al menos 15 puntos):")
print(s[["artefacto", "clase", "pct", "ic95_inf", "ic95_sup", "dif_vs_nv_pp", "p_vs_nv"]].to_string(index=False) if len(s) else "  ninguna")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(pct.values, cmap="Blues", vmin=0, vmax=max(50, np.nanmax(pct.values)), aspect="auto")
ax.set_xticks(range(len(CLASES)), CLASES)
ax.set_yticks(range(len(ARTEFACTOS)), [ARTEFACTOS[a][0] for a in ARTEFACTOS]); ax.grid(False)
senal = tab.set_index(["artefacto", "clase"]).senal
for i, a in enumerate(ARTEFACTOS):
    for j, c in enumerate(CLASES):
        v = pct.loc[a, c]
        ax.text(j, i, es(v, 0) + ("*" if senal.get((a, c), False) else ""), ha="center", va="center",
                fontsize=9, color="white" if v > 30 else TINTA, fontweight="bold" if senal.get((a, c), False) else None)
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01); cb.outline.set_visible(False)
cb.set_label("% de imágenes de la clase", color=TINTA2)
guardar(fig, "fig_artefactos_por_clase")
print("* = diferencia frente a nv de al menos 15 puntos y p < 0,05 (prueba exacta de Fisher)")

## 7. Acuerdo entre anotadores (si hubo segundo anotador)

In [ ]:
otros = [p for p in OUT.glob("etiquetas_*.csv") if p.name != "etiquetas_adrian.csv"]
kappas = None
if not otros:
    print("No hay segundo anotador: esta sección se omite (se declara como limitación).")
for p in otros:
    b = cargar_etiquetas(p, muestra)
    quien = p.stem.replace("etiquetas_", "")
    if len(b) < len(ACUERDO_IDS):
        print(f"[!] {quien} lleva {len(b)} de {len(ACUERDO_IDS)} imágenes: espera a que termine antes de esta sección.")
        continue
    kappas = acuerdo(et, b)
    kappas.to_csv(OUT / f"acuerdo_{p.stem.replace('etiquetas_', '')}.csv")
    print(f"Adrián vs {p.stem.replace('etiquetas_', '')}:"); display(kappas)
print("Referencia (Landis y Koch): kappa < 0,4 bajo · 0,4–0,6 moderado · 0,6–0,8 sustancial · > 0,8 casi perfecto")

## 8. Detección automática del borde negro
En la Etapa 2 se midió el brillo de las esquinas de las 10.015 imágenes. Aquí se comprueba si la regla **"esquinas < 40"** coincide con tu etiqueta manual de *borde negro*. Si la regla es fiable, se aplica a **todas** las imágenes de train y se obtiene la frecuencia del borde negro por clase con mucha más muestra que 50.

In [ ]:
color = pd.read_csv(DRIVE_ROOT / "results" / "m1" / "etapa2" / "color_por_imagen.csv")   # medidas de la Etapa 2
val = pd.DataFrame([validar_regla_borde(etiq, color, u) for u in [30, 40, 50, 60, 80]])
display(val)
mejor = val.assign(f1=2 * val.precision * val.sensibilidad / (val.precision + val.sensibilidad)).sort_values("f1", ascending=False).iloc[0]
UMBRAL = int(mejor.umbral)
print(f"Umbral con mejor equilibrio en la muestra: {UMBRAL} (precisión {mejor.precision}, sensibilidad {mejor.sensibilidad})")

tr = split[split.split == "train"].merge(color[["image_id", "esquinas"]], on="image_id")
cob = (tr.assign(borde_auto=tr.esquinas < UMBRAL).groupby("dx").borde_auto.agg(["sum", "size", "mean"])
         .reindex(CLASES).rename(columns={"sum": "con_borde", "size": "n_train", "mean": "pct"}))
cob["pct"] = (100 * cob.pct).round(1)
cob.to_csv(OUT / "borde_negro_auto_train.csv"); display(cob)
print("Fiable para el Módulo 3 si precisión y sensibilidad ≥ 0,8; si no, solo vale la muestra manual.")

## 9. Entrega al Módulo 3 y registro en MLflow

In [ ]:
resumen = {
    "muestra": {"n": len(muestra), "por_clase": 50, "origen": "train", "una_foto_por_lesion": True, "semilla": 42,
                "a_ciegas": True},
    "definiciones": {a: d for a, (_, d) in ARTEFACTOS.items()},
    "pct_por_clase": pct.round(1).to_dict(),
    "senales": tab[tab.senal][["artefacto", "clase", "pct", "dif_vs_nv_pp"]].to_dict("records"),
    "regla_borde": {"umbral": UMBRAL, "precision": float(mejor.precision), "sensibilidad": float(mejor.sensibilidad)},
    "kappa": None if kappas is None else kappas.kappa.to_dict(),
}
(OUT / "resumen_artefactos.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=str))

import mlflow
from src.tracking import setup_mlflow, standard_tags
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
setup_mlflow(cfg); mlflow.set_experiment(cfg["mlflow"]["experiments"]["m1"])
shutil.copy(SPLIT_DRIVE, cfg["paths"]["split_csv"])     # para que el tag split_sha256 lo lea del repo
tags = standard_tags(cfg, modulo="m1", fase="artefactos", autor="adrian", dataset_sha256=DATASET_SHA256_ESPERADO)
assert tags["split_sha256"] == SPLIT_SHA256_ESPERADO
with mlflow.start_run(run_name="m1-artefactos", tags=tags) as run:
    mlflow.log_params({"n_muestra": len(muestra), "por_clase": 50, "semilla_muestra": 42,
                       "a_ciegas": True, "umbral_borde": UMBRAL, "segundo_anotador": bool(otros)})
    mlflow.log_metrics({f"pct_{a}_{c}": float(pct.loc[a, c]) for a in ARTEFACTOS for c in CLASES})
    mlflow.log_artifacts(str(OUT), artifact_path="etapa3")
    mlflow.log_artifacts(str(FIG), artifact_path="figuras")
print("[+] Run m1-artefactos registrado:", run.info.run_id)
print("Para el Módulo 3: TFM/results/m1/etapa3/ (artefactos_muestra.csv, artefactos_por_clase.csv, resumen_artefactos.json)")